## 1. Load Data for configuration and Initialize and setup Great Expectations

This is for checking data quality for the bronze layer only to see if data is even ready for DataCleaning in the silver layer

In [2]:
# prepare a Spark Dataframe and fill it with the to be checked data from the Lakehouse Bronze Layer
df_bronze_conf = spark.sql("SELECT * FROM lh_CustomerChurn.bronze.CustomerChurn LIMIT 100")

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 6, Finished, Available, Finished, False)

In [3]:
# Delete unused Columns
df_bronze_conf = df_bronze_conf.drop('cr8b0_monthlycharges_base','cr8b0_totalcharges_base','createdby', 'createdbyname', 'createdbyyominame', 'createdon', 
            'createdonbehalfby', 'createdonbehalfbyname', 'createdonbehalfbyyominame', 
            'exchangerate', 'importsequencenumber', 'modifiedby', 'modifiedbyname', 'modifiedbyyominame',
            'modifiedon', 'modifiedonbehalfby', 'modifiedonbehalfbyname', 'modifiedonbehalfbyyominame', 
            'overriddencreatedon', 'ownerid', 'owneridname', 'owneridtype', 'owneridyominame', 'owningbusinessunit',
            'owningbusinessunitname', 'owningteam', 'owninguser', 'statecode', 'statuscode', 'timezoneruleversionnumber', 
            'transactioncurrencyid', 'transactioncurrencyidname', 'utcconversiontimezonecode', 'versionnumber'
            )

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 7, Finished, Available, Finished, False)

In [4]:
# import great_expectations using workspace settings -> Library Management
from pyspark.sql import SparkSession
import great_expectations as gx

# initialize Spark session
#spark = SparkSession.builder.appName("MedallionGX").get_session()

# Initialize GX for configuration & set up in-memory source
context = gx.get_context(mode="file")
# declare a spark source as a datasource
datasource = context.data_sources.add_or_update_spark(name = "gx1_datasource")
# declare a data asset   
data_asset = datasource.add_dataframe_asset(name = "gx1_df_asset")
# build a batch request for testing, passing in the loaded dataframe
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_def1")


StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 8, Finished, Available, Finished, False)

## 2. Configure Expectations suite to be called over runtime later

In [5]:
# add an (empty) expectation suite or overrite existing suite
gx_suite = gx.ExpectationSuite(name="expectation_suite_One")
gx_suite = context.suites.add_or_update(gx_suite)

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 9, Finished, Available, Finished, False)

### 2.1 Define Table Level expectations

In [6]:
# -- 2.1. Define table level expectations
columns = list(df_bronze_conf.columns)
expectation = gx.expectations.ExpectTableColumnsToMatchSet(column_set=columns)
gx_suite.add_expectation(expectation)


StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 10, Finished, Available, Finished, False)

ExpectTableColumnsToMatchSet(id='bc797b21-d3ad-4da3-b321-87547ffd546f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column_set=['cr8b0_churnstatus', 'cr8b0_contracttype', 'cr8b0_customerid', 'cr8b0_customersubscriptionid', 'cr8b0_deviceprotectionenabled', 'cr8b0_gender', 'cr8b0_hasdependents', 'cr8b0_haspartner', 'cr8b0_internetservicetype', 'cr8b0_monthlycharges', 'cr8b0_multiplephonelines', 'cr8b0_onlinebackupenabled', 'cr8b0_onlinesecurityenabled', 'cr8b0_paperlessbilling', 'cr8b0_paymentmethod', 'cr8b0_phoneserviceenabled', 'cr8b0_recordid', 'cr8b0_seniorcitizenstatus', 'cr8b0_streamingmoviesenabled', 'cr8b0_streamingtvenabled', 'cr8b0_techsupportenabled', 'cr8b0_tenuremonths', 'cr8b0_totalcharges'], exact_match=True)

### 2.2. Define column level expectations

In [ ]:
# 1. Ensure Churnstatus is either True or False
expectation1 = gx.expectations.ExpectColumnValuesToBeOfType(
    column="cr8bO_contracttype",
    type_="BooleanType"
)
gx_suite.add_expectation(expectation1)

In [ ]:
expectation2 = gx.expectations.ExpectColumnValuesToBeOfType(
        column="cr8bO_customerid", 
        type_="StringType"
    )
gx_suite.add_expectation(expectation2)

## 2.3. Evaluate results on test dataset

In [7]:
batch_parameters = {"dataframe": df_bronze_conf}
batch = batch_definition.get_batch(batch_parameters=batch_parameters)
validation_results = batch.validate(gx_suite)

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 11, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 12, Finished, Available, Finished, False)

In [8]:
validation_results

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 13, Finished, Available, Finished, False)

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_table_columns_to_match_set",
        "kwargs": {
          "batch_id": "gx1_datasource-gx1_df_asset",
          "column_set": [
            "cr8b0_churnstatus",
            "cr8b0_contracttype",
            "cr8b0_customerid",
            "cr8b0_customersubscriptionid",
            "cr8b0_deviceprotectionenabled",
            "cr8b0_gender",
            "cr8b0_hasdependents",
            "cr8b0_haspartner",
            "cr8b0_internetservicetype",
            "cr8b0_monthlycharges",
            "cr8b0_multiplephonelines",
            "cr8b0_onlinebackupenabled",
            "cr8b0_onlinesecurityenabled",
            "cr8b0_paperlessbilling",
            "cr8b0_paymentmethod",
            "cr8b0_phoneserviceenabled",
            "cr8b0_recordid",
            "cr8b0_seniorcitizenstatus",
            "cr8b0_streamingmoviesenabled",
            "cr8b0_streamingtvenabled",


# 3. Bundle suite and batch into validation definition and checkpoint w/ bundled actions for easy execution later

In [9]:
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite=gx_suite,
    name="gx_bronze_validation",
)
_ = context.validation_definitions.add(validation_definition)

action_list = [
    gx.checkpoint.UpdateDataDocsAction(
        name="update_all_data_docs",
    ),
]
checkpoint = gx.Checkpoint(
    name="gx_checkpoint_1",
    validation_definitions=[validation_definition],
    actions=action_list,
    result_format={
        "result_format": "COMPLETE",
    },
)
_ = context.checkpoints.add(checkpoint)


StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 14, Finished, Available, Finished, False)

In [10]:
validation_results = checkpoint.run(
    batch_parameters=batch_parameters,
    expectation_parameters=gx_suite.expectations,
)

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 15, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 16, Finished, Available, Finished, False)

In [11]:
# -- 4. Run checkpoint to validate if everything works properly
file_identifier = "abfss://21a5c954-bf7b-4072-b2e6-389ef4287915@onelake.dfs.fabric.microsoft.com/86585eb0-bf45-4c35-8892-3892ddd9b3bb/Tables/bronze/CustomerChurn".split("/")[-1]
runid = gx.RunIdentifier(run_name=f"Configuration run - {file_identifier}")
results = checkpoint.run(batch_parameters=batch_parameters, run_id=runid)

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 17, Finished, Available, Finished, False)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 18, Finished, Available, Finished, False)

In [12]:
# shell script to copy the great expectations folder into the root directory in the lakehouse files area
!cp -r gx/ /lakehouse/default/Files/

StatementMeta(, 45a1fda2-f4a5-4bea-b258-16c67b008ae1, 19, Finished, Available, Finished, False)